In [ ]:
!pip install -U ultralytics albumentations==1.4.3 scipy==1.11.4 numpy==1.26.4


# <p style='font-family: Signika+Negative; background-color:#E1BEE7; font-weight:bold; color:#6A1B9A; border:3px solid #6A1B9A; border-radius:10px; box-shadow:0px 3px 10px rgba(0,0,0,0.2); padding:10px; text-align:center;'>✨ Settings ✨</p>


In [ ]:
from ultralytics import YOLO
import os
import shutil
import random
import yaml
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as patches
import pandas as pd
import numpy as np

# <p style='font-family: Signika+Negative; background-color:#E1BEE7; font-weight:bold; color:#6A1B9A; border:3px solid #6A1B9A; border-radius:10px; box-shadow:0px 3px 10px rgba(0,0,0,0.2); padding:10px; text-align:center;'>✨ Data preprocessing and visualization ✨</p>


In [ ]:
base_dir = "/kaggle/input/drone-dataset-uav/drone_dataset_yolo/dataset_txt"
images = [f for f in os.listdir(base_dir) if f.endswith(".jpg")]

train_split = 0.75
val_split = 0.15
test_split = 0.1

random.shuffle(images)
n = len(images)
train_imgs = images[:int(n*train_split)]
val_imgs = images[int(n*train_split):int(n*(train_split+val_split))]
test_imgs = images[int(n*(train_split+val_split)):]

def copy_pairs(img_list, dest_folder):
    for img in img_list:
        label = img.replace(".jpg", ".txt")
        shutil.copy(os.path.join(base_dir, img), os.path.join(f"dataset/images/{dest_folder}", img))
        if os.path.exists(os.path.join(base_dir, label)):
            shutil.copy(os.path.join(base_dir, label), os.path.join(f"dataset/labels/{dest_folder}", label))

for split in ["train", "val", "test"]:
    os.makedirs(f"dataset/images/{split}", exist_ok=True)
    os.makedirs(f"dataset/labels/{split}", exist_ok=True)

copy_pairs(train_imgs, "train")
copy_pairs(val_imgs, "val")
copy_pairs(test_imgs, "test")

print("Everything is going as espected!")

In [ ]:
for root, dirs, files in os.walk("dataset"):
    print(root, len(files), "files")


In [ ]:
import yaml

data = {
    'train': 'dataset/images/train',
    'val': 'dataset/images/val',
    'test': 'dataset/images/test',
    'nc': 1,
    'names': ['drone']
}

with open('data.yaml', 'w') as f:
    yaml.dump(data, f, sort_keys=False)

print("File data.yaml created!")

In [ ]:
img_dir = "dataset/images/train"
label_dir = "dataset/labels/train"

augmentor = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(p=0.5),
    A.RandomGamma(p=0.3),
    A.Rotate(limit=15, p=0.4, border_mode=cv2.BORDER_REFLECT_101),
    A.Blur(blur_limit=3, p=0.2),
    A.HueSaturationValue(p=0.3)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

num_aug = 1 

def clip_boxes(bboxes):
    clipped = []
    for box in bboxes:
        x, y, w, h = [max(0, min(1, v)) for v in box]
        clipped.append([x, y, w, h])
    return clipped

for img_name in os.listdir(img_dir):
    if not img_name.endswith(".jpg"):
        continue

    img_path = os.path.join(img_dir, img_name)
    label_path = os.path.join(label_dir, img_name.replace(".jpg", ".txt"))

    image = cv2.imread(img_path)
    if image is None or not os.path.exists(label_path):
        continue

    with open(label_path) as f:
        boxes, classes = [], []
        for line in f.readlines():
            vals = line.strip().split()
            if len(vals) != 5:
                continue
            cls, x, y, bw, bh = map(float, vals)
            boxes.append([x, y, bw, bh])
            classes.append(int(cls))

    for i in range(num_aug):
        try:
            augmented = augmentor(image=image, bboxes=boxes, class_labels=classes)
        except ValueError:
            continue

        aug_img = augmented["image"]
        aug_boxes = clip_boxes(augmented["bboxes"])
        aug_classes = augmented["class_labels"]

        new_name = img_name.replace(".jpg", f"_aug{i}.jpg")
        cv2.imwrite(os.path.join(img_dir, new_name), aug_img)

        with open(os.path.join(label_dir, new_name.replace(".jpg", ".txt")), "w") as f:
            for cls, (x, y, bw, bh) in zip(aug_classes, aug_boxes):
                f.write(f"{cls} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n")

print("✅ Augmentation completed successfully!")

# <p style='font-family: Signika+Negative; background-color:#E1BEE7; font-weight:bold; color:#6A1B9A; border:3px solid #6A1B9A; border-radius:10px; box-shadow:0px 3px 10px rgba(0,0,0,0.2); padding:10px; text-align:center;'>✨ Eda✨</p>


In [ ]:
splits = ["train", "val", "test"]
for split in splits:
    n_imgs = len([f for f in os.listdir(f"dataset/images/{split}") if f.endswith(".jpg")])
    n_labels = len([f for f in os.listdir(f"dataset/labels/{split}") if f.endswith(".txt")])
    print(f"{split}: {n_imgs} immagini, {n_labels} label")


In [ ]:
widths, heights = [], []

for label_file in os.listdir("dataset/labels/train"):
    if not label_file.endswith(".txt"):
        continue
    with open(os.path.join("dataset/labels/train", label_file)) as f:
        for line in f:
            cls, x, y, w, h = map(float, line.strip().split())
            widths.append(w)
            heights.append(h)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.hist(widths, bins=30, color='skyblue')
plt.title("Distribution of bbox widths (normalized)")

plt.subplot(1,2,2)
plt.hist(heights, bins=30, color='salmon')
plt.title("Distribution of bbox heights (normalized)")

plt.show()


In [ ]:
def show_image_with_bbox(split="train"):
    img_file = random.choice(os.listdir(f"dataset/images/{split}"))
    if not img_file.endswith(".jpg"):
        return
    img_path = f"dataset/images/{split}/{img_file}"
    label_path = f"dataset/labels/{split}/{img_file.replace('.jpg','.txt')}"

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with open(label_path) as f:
        for line in f:
            cls, x, y, w, h = map(float, line.strip().split())
            h_img, w_img, _ = img.shape
            x1 = int((x - w/2) * w_img)
            y1 = int((y - h/2) * h_img)
            x2 = int((x + w/2) * w_img)
            y2 = int((y + h/2) * h_img)
            cv2.rectangle(img, (x1, y1), (x2, y2), (255,0,0), 2)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

show_image_with_bbox("train")
show_image_with_bbox("train")
show_image_with_bbox("train")

# <p style='font-family: Signika+Negative; background-color:#E1BEE7; font-weight:bold; color:#6A1B9A; border:3px solid #6A1B9A; border-radius:10px; box-shadow:0px 3px 10px rgba(0,0,0,0.2); padding:10px; text-align:center;'>✨ Train the model ✨</p>


In [ ]:
model = YOLO('yolov8l.pt')

model.train(
    data='data.yaml',
    epochs=70,
    batch=16,
    imgsz=512,
    augment=True,
    project='drone_detection',
    name='yolov8_drone',
    exist_ok=True,
    save=True,
    save_period=5,
    patience=8,
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.1,
    momentum=0.9,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    cache='ram',
    device='cuda',
    workers=8,
    deterministic=True,
    val=True,
    plots=True,
    amp=True,
    multi_scale=False,
    close_mosaic=10
)


In [ ]:
csv_path = "drone_detection/yolov8_drone/results.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    
    table = df[['epoch', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
                'metrics/precision(B)', 'metrics/recall(B)',
                'metrics/mAP50(B)', 'metrics/mAP50-95(B)']]

    table = table.groupby('epoch', as_index=False).mean()
    display(table)
else:
    print("results.csv not found. Make sure training is finished and path is correct.")


# YOLOv8 Drone Detection

This project trains a **YOLOv8 model** (`yolov8`) to detect drones in images. The dataset is divided into **train, validation, and test splits**, and augmentation is applied both manually and during training.

## Training Configuration

- **Model:** `yolov8l.pt` (nano version, pre-trained)
- **Image size:** 512x512
- **Batch size:** 16
- **Epochs:** 70
- **Optimizer:** AdamW
- **Learning rate:** 0.0005
- **Early stopping patience:** 8 epochs
- **Mixed precision:** Enabled (`amp=True`)
- **Device:** GPU (`cuda`)
- **Checkpoint saving:** every 5 epochs
- **Mosaic:** Disabled in last 10 epochs

## Training Metrics

| Term       | Meaning                                                   | Notes / Tips                                         |
|-----------|-----------------------------------------------------------|-----------------------------------------------------|
| Epoch      | One full pass through the entire training dataset        | More epochs = more learning time; watch for overfitting |
| GPU_mem    | Amount of GPU memory used during training (GB)           | Helps assess if batch size or model is too large   |
| box_loss   | Loss related to the position and size of bounding boxes | Should decrease over time; lower is better         |
| cls_loss   | Classification loss for object classes                  | Lower values mean better classification accuracy   |
| dfl_loss   | Distribution Focal Loss for bounding box precision      | Should also decrease over training                 |
| Instances  | Number of objects detected in the batch                 | Indicates how many objects the model sees per batch |
| Size       | Input image size                                        | Balances between speed and accuracy                |

### Validation Metrics per Epoch

| Term      | Meaning                                                | Ideal Values / Notes                               |
|-----------|--------------------------------------------------------|---------------------------------------------------|
| Class     | Object class/category                                  | e.g., drone                                      |
| Images    | Number of images used in validation                   |                                                   |
| Instances | Total number of annotated objects in validation set   |                                                   |
| P         | Precision: fraction of correct positive detections    | Closer to 1 is better                             |
| R         | Recall: fraction of actual objects detected          | Closer to 1 is better                             |
| mAP50     | Mean Average Precision at IoU=0.5                     | Closer to 1 is better                             |
| mAP50-95  | Mean Average Precision averaged over IoU 0.5–0.95     | Higher is better; more robust evaluation         |

### Simple Explanation

- **Loss:** How much the model is currently "wrong" during training (lower is better)
- **Precision:** Of all objects the model predicted, how many were actually correct
- **Recall:** Of all the real objects present, how many did the model find
- **mAP (Mean Average Precision):** Combines precision and recall into a single performance metric, higher is better


# <p style='font-family: Signika+Negative; background-color:#E1BEE7; font-weight:bold; color:#6A1B9A; border:3px solid #6A1B9A; border-radius:10px; box-shadow:0px 3px 10px rgba(0,0,0,0.2); padding:10px; text-align:center;'>✨ Test the model ✨</p>

In [ ]:
test_metrics = model.val(
    data='/kaggle/working/data.yaml',
    split='test',
    imgsz=512,
    batch=8,
    save=False,
    verbose=False
)

precision, recall, map50, map50_95 = test_metrics.mean_results()

print("\n-------------------------------------\n")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"mAP50: {map50:.3f}")
print(f"mAP50-95: {map50_95:.3f}")
print("\n-------------------------------------\n")

# <p style='font-family: Signika+Negative; background-color:#E1BEE7; font-weight:bold; color:#6A1B9A; border:3px solid #6A1B9A; border-radius:10px; box-shadow:0px 3px 10px rgba(0,0,0,0.2); padding:10px; text-align:center;'>✨ Visualize some examples ✨</p>


In [ ]:
TEST_IMAGES_DIR = "dataset/images/test"

def show_inference(img_path, model, ax, class_names):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = model.predict(img_path)
    
    boxes = results[0].boxes.xyxy.cpu().numpy()
    scores = results[0].boxes.conf.cpu().numpy()
    classes = results[0].boxes.cls.cpu().numpy().astype(int)

    ax.imshow(img_rgb)
    ax.axis('off')
    ax.set_title(os.path.basename(img_path), fontsize=12, weight='bold')

    colors = plt.cm.get_cmap('tab20', len(class_names))

    for box, score, cl in zip(boxes, scores, classes):
        x1, y1, x2, y2 = box
        width, height = x2 - x1, y2 - y1
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2, edgecolor=colors(cl), facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x1, max(0, y1 - 5),
            f'{class_names[cl]} {score:.2f}',
            color='white', fontsize=10, weight='bold',
            backgroundcolor='black', alpha=0.6
        )

test_imgs = os.listdir(TEST_IMAGES_DIR)
random_imgs = random.sample(test_imgs, min(4, len(test_imgs)))

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
for img_path, ax in zip(random_imgs, axs.flatten()):
    full_path = os.path.join(TEST_IMAGES_DIR, img_path)
    show_inference(full_path, model, ax, data['names'])

plt.tight_layout()
plt.show()

In [ ]:
model.save('final_drone_model.pt') 